### dbGap FHIR Notebooks - Exercise 3

## Learning Objectives and Key Concepts

In this exercise, you will query the dbGaP FHIR test study: phs002409: 


- Recreate subject data as submitted 
- Aggregate FHIR Observations for a subject
- Enable Dataframe queries 
- Explore the same queries via FHIR

## Motivation/Purpose
To explore challenges in different representations of dbGaP data

## Requires
- ipywidgets
- pandas

 ### Icons in this Guide
 📘 A link to a useful external reference related to the section the icon appears in  

 🖐 A hands-on section where you will code something or interact with the server  
 
 
Acknowledging use of code snippets from [NIH FHIR training](https://github.com/NIH-ODSS/fhir-exercises/tree/main/Python) Exercise 0.



In [ ]:
from dbgap_fhir import DbGapFHIR

FHIR_SERVER = "https://dbgap-api.ncbi.nlm.nih.gov/fhir/x1"
mf = DbGapFHIR(FHIR_SERVER)

### Query subjects for the study

First some basic exploration of the data in the study is helpful via FHIR.

Find all the patients registered as subjects to the study.

In [ ]:
study_id = "phs002409"
patients = mf.run_query(
    f"Patient?_has:ResearchSubject:individual:study={study_id}"
)

In [ ]:
patients[812]

### Look at the study details

In [ ]:
studies = mf.run_query(f"ResearchStudy?_id={study_id}")

In [ ]:
studies[0]

### Observations
The query below was simply copied from the Kids First examples. It seems reasonable but is not yet implemented in dbGaP FHIR.

Patient?_has:ResearchSubject:individual:study={study_id}&_revinclude=Observation:subject"

We use instead a workaround to run a query for the Observations for each Patient.

The following creates a dataframe showing the observations for each patient.
It also
* saves the definition of each observation (column) to a file
* checks that the definition is the same for each instance of the observation are the same.

This is also an opportunity to illustrate the use of iPython widgets to display a progress bar.

📘 You can find out more about progress bars and other iPython widgets [here](https://ipywidgets.readthedocs.io/en/stable/).

Because we are going to make a large number of requests in a short period of time this is a good example of a task where having an api key is helpful.

📘See notebook 2 for details.

In [ ]:
import os

API_KEY_PATH = "~/.keys/ncbi_api_key.txt"

# the os.path.expanduser expands file paths which include ~/ representation for the user's home directory
with open(os.path.expanduser(API_KEY_PATH)) as f:
    api_key = f.read()

    mf = DbGapFHIR(FHIR_SERVER, api_key=api_key)

### Now we can run the query
In fact one query per patient. It will take some time. The progress bar will do what progress bars do.

In [ ]:
from ipywidgets import IntProgress
from IPython.display import display

prog = IntProgress(min=0, max=len(patients))  # instantiate the bar
display(prog)  # display the bar

all_obs = []
patients_with_obs = []
for p in patients:
    # print(p['id'])
    obs = mf.run_query(f"Observation?subject={p['id']}", show_stats=False)
    if len(obs) > 0:
        all_obs += obs
        patients_with_obs.append(p)
    prog.value += 1

print(f"{len(patients_with_obs)} patients had observations")
print(f"{len(all_obs)} total observations")

### Construct a dataframe

In [ ]:
import json
import pandas as pd
from collections import Counter


patient_observations_dict = {}
variable_definitions = {}
observations = []
obsCounter = Counter()
codeCounter = Counter()
vccCounter = Counter()
printObsCounts = True
rlimit = 20
nn = 0
for r in all_obs:

    if r["resourceType"] == "Observation":
        # print(json.dumps(r,indent=3))
        # nn+=1
        # if nn > rlimit:
        #    break
        subject_id = r["subject"]["reference"]
        obsCounter[subject_id] += 1
        obs_display_name = r["code"]["coding"][0]["display"]
        if "valueQuantity" in r:
            value_text = r["valueQuantity"]["value"]
            # value_unit = r['valueQuantity']['unit']
        elif "valueCodeableConcept" in r:
            value_text = r["valueCodeableConcept"]["coding"][0]["display"]
        else:
            value_text = "unknown"
        codeCounter[obs_display_name] += 1
        # vccCounter[vcc_text] +=1
        observations.append(r)

        if subject_id not in patient_observations_dict:
            patient_observations_dict[subject_id] = {
                obs_display_name: value_text
            }
        else:
            patient_observations_dict[subject_id][obs_display_name] = value_text

# Summarize
print(f"Number of patients with observations {len(obsCounter.keys())}")

if printObsCounts:
    print("Observation count per patient")
    print(json.dumps(obsCounter, indent=3))
# print("Coding counts")
# print(json.dumps(codeCounter, indent=3))
df = pd.DataFrame.from_dict(codeCounter, orient="index")

### Create and display the Dataframe

In [ ]:
pd.set_option("display.max_rows", 30, "display.max_columns", None)
patient_df = pd.DataFrame.from_dict(patient_observations_dict, orient="index")
# patient_df.fillna('', inplace=True)
display(patient_df)

Each Observation will represent one cell in the above table.

813 rows × 51 columns = 41463

🖐 Note that is not the same total 41224 listed at the end of our queries. You may want to explore why. NaN is not the only reason for the difference.

Save the table above to a file

🖐 subtitute the filename below as you wish

In [ ]:
txt_file_path = "results/phs002409_workaround_obs.txt"
patient_df.to_csv(txt_file_path, sep="\t")

### Query on observation value

#### Text value
Exercise to do - Formulate the query that would identify where the value of ENV_SMOKE is 'yes'

Ahead of doing that we can check by independent<sup>1</sup> means which patients we should get back from such a query.

The filter below on the DataFrame identfies 6 patients which match these criteria.

In [ ]:
patient_df[patient_df.ENV_SMOKE == "1"]

#### Numeric value
Exercise to do - write the FHIR query that would identify patients from this study where the PREFEVPP_baseline is greater than 120.

Again we can use our DataFrame for an independent<sup>1</sup> test to indicate which Patients we would expect to result from such a FHIR query

In [ ]:
patient_df[patient_df.PREFEVPP_baseline > 120]

<sup>1</sup> Neither of the tests above is a completely indpendent test, as the DataFrame was itself generated via the FHIR API. A better test would rely on an independent means of accessing the source data.

### Queries on specific Observation values

In [ ]:
vals = mf.run_query(
    "Observation?combo-code-value-quantity=phv00492057.v1.p1$gt1"
)

In [ ]:
vals = mf.run_query(
    "Observation?combo-code-value-quantity=PX091601370000$gt150"
)